# 数据集采样与结构查看

这个 notebook 用于快速抽样查看 GRID 数据集的原始 TFRecord row 结构，以及可选查看经过 Hydra data preprocessing 后的样本结构。

默认数据目录约定：

- item 级输入：`{DATA_DIR}/items`
- 训练：`{DATA_DIR}/training`
- 验证：`{DATA_DIR}/evaluation`
- 测试/预测：`{DATA_DIR}/testing`

使用方式：修改下一格中的 `DATA_DIR`、`SPLIT` 和采样数量后，逐格执行。

In [2]:
from __future__ import annotations

import itertools
import json
from pathlib import Path
from pprint import pprint
from typing import Any

import numpy as np
import torch
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf
import rootutils

# 从仓库根目录运行 notebook 时通常不需要修改。
PROJECT_ROOT = rootutils.setup_root(search_from=Path(), indicator="pyproject.toml", pythonpath=True)
DATA_DIR = PROJECT_ROOT / "data" / "beauty"
CONFIG_DIR = PROJECT_ROOT / "configs"

# 可选：items / training / evaluation / testing。
SPLIT = "training"

# 原始 row 采样数量。
NUM_RAW_SAMPLES = 5

# 预处理后样本采样数量。
NUM_PREPROCESSED_SAMPLES = 5


## 工具函数

下面的函数用于发现 TFRecord 文件、截断大对象展示，并递归汇总 Python / NumPy / Torch 对象结构。

In [3]:
def find_tfrecord_files(data_dir: Path, split: str, limit: int | None = None) -> list[str]:
    split_dir = data_dir / split
    files = sorted(split_dir.glob("*.tfrecord.gz"))
    if not files:
        raise FileNotFoundError(f"No *.tfrecord.gz files found under {split_dir.resolve()}")
    files = files[:limit] if limit is not None else files
    return [str(path) for path in files]


def preview_value(value: Any, max_items: int = 8, max_text: int = 120) -> Any:
    if isinstance(value, torch.Tensor):
        flat = value.detach().cpu().reshape(-1)
        return {
            "type": "torch.Tensor",
            "shape": tuple(value.shape),
            "dtype": str(value.dtype),
            "preview": flat[:max_items].tolist(),
        }
    if isinstance(value, np.ndarray):
        flat = value.reshape(-1)
        return {
            "type": "np.ndarray",
            "shape": value.shape,
            "dtype": str(value.dtype),
            "preview": flat[:max_items].tolist(),
        }
    if isinstance(value, bytes):
        text = value[:max_text]
        try:
            text = text.decode("utf-8", errors="replace")
        except Exception:
            text = repr(text)
        return {"type": "bytes", "len": len(value), "preview": text}
    if isinstance(value, str):
        return value[:max_text]
    if isinstance(value, dict):
        return {k: preview_value(v, max_items=max_items, max_text=max_text) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [preview_value(v, max_items=max_items, max_text=max_text) for v in list(value)[:max_items]]
    return value


def describe_structure(obj: Any, depth: int = 0, max_depth: int = 4) -> Any:
    if depth >= max_depth:
        return type(obj).__name__
    if isinstance(obj, torch.Tensor):
        return {"type": "torch.Tensor", "shape": tuple(obj.shape), "dtype": str(obj.dtype)}
    if isinstance(obj, np.ndarray):
        return {"type": "np.ndarray", "shape": obj.shape, "dtype": str(obj.dtype)}
    if isinstance(obj, dict):
        return {k: describe_structure(v, depth + 1, max_depth) for k, v in obj.items()}
    if hasattr(obj, "__dataclass_fields__"):
        return {
            "type": type(obj).__name__,
            "fields": {
                name: describe_structure(getattr(obj, name), depth + 1, max_depth)
                for name in obj.__dataclass_fields__
            },
        }
    if isinstance(obj, (list, tuple)):
        return {"type": type(obj).__name__, "len": len(obj), "items": [describe_structure(v, depth + 1, max_depth) for v in obj[:3]]}
    return {"type": type(obj).__name__, "repr": repr(obj)[:120]}

## 1. 查看原始 TFRecord row

这一步直接使用项目内 `TFRecordReader` 读取 `{DATA_DIR}/{SPLIT}` 下的 `*.tfrecord.gz`，不执行任何 preprocessing。

In [5]:
from src.data.components.readers import TFRecordReader

raw_files = find_tfrecord_files(DATA_DIR, SPLIT, limit=3)
print(f"Using {len(raw_files)} file(s):")
for file in raw_files:
    print(" -", file)

raw_reader = TFRecordReader(list_of_file_paths=raw_files, shuffle_rows=False)
raw_samples = list(itertools.islice(raw_reader.iterrows(), NUM_RAW_SAMPLES))
print(f"Loaded {len(raw_samples)} raw sample(s).")

Using 3 file(s):
 - E:\projects\GRID\data\beauty\training\partition_0.tfrecord.gz
 - E:\projects\GRID\data\beauty\training\partition_1.tfrecord.gz
 - E:\projects\GRID\data\beauty\training\partition_10.tfrecord.gz
Loaded 5 raw sample(s).


In [9]:
print("Raw sample structure:")
pprint(describe_structure(raw_samples[0]))

print("\nRaw sample preview:")
pprint(preview_value(raw_samples[0]))

Raw sample structure:
{'embedding': {'dtype': 'float32', 'shape': (3072,), 'type': 'np.ndarray'},
 'sequence_data': {'dtype': 'int64', 'shape': (4,), 'type': 'np.ndarray'},
 'text': {'dtype': '|S248', 'shape': (4,), 'type': 'np.ndarray'},
 'user_id': {'dtype': 'int64', 'shape': (1,), 'type': 'np.ndarray'}}

Raw sample preview:
{'embedding': {'dtype': 'float32',
               'preview': [-0.018911147490143776,
                           0.0584166944026947,
                           0.01738640107214451,
                           -0.03140246868133545,
                           -0.029518403112888336,
                           -0.06503795832395554,
                           -0.0850256159901619,
                           0.018116170540452003],
               'shape': (5376,),
               'type': 'np.ndarray'},
 'sequence_data': {'dtype': 'int64',
                   'preview': [3, 11, 12, 13, 14, 15, 16],
                   'shape': (7,),
                   'type': 'np.ndarray'},
 '

## 2. 查看某个 experiment 的 preprocessing 后样本

这一步会通过 Hydra 组合 data config，并实例化对应 dataset config 中声明的 preprocessing functions。

注意：像 `tiger_train` 这类配置需要 `semantic_id_path`、`num_hierarchies` 等必填参数；如果你只想看原始结构，可以跳过本节。

In [ ]:
# 可选 experiment：sem_embeds_inference / tiger_train / tiger_inference / rkmeans_train 等。
EXPERIMENT = "tiger_train"

# tiger_train / tiger_inference 等需要 semantic_id_path；不需要时可保留 None。
SEMANTIC_ID_PATH: str | None = None
NUM_HIERARCHIES: int | None = 3
DEVICES = 1

overrides = [
    f"experiment={EXPERIMENT}",
    f"data_dir={DATA_DIR.as_posix()}",
    f"devices={DEVICES}",
]
if SEMANTIC_ID_PATH is not None:
    overrides.append(f"semantic_id_path={SEMANTIC_ID_PATH}")
if NUM_HIERARCHIES is not None:
    overrides.append(f"num_hierarchies={NUM_HIERARCHIES}")

with initialize_config_dir(config_dir=str(CONFIG_DIR), version_base="1.3"):
    cfg = compose(config_name="main", overrides=overrides)

print(OmegaConf.to_yaml(cfg.data, resolve=False)[:4000])

In [ ]:
# 选择 dataloader 配置：train_dataloader / val_dataloader / test_dataloader / predict_dataloader。
DATALOADER_CONFIG_NAME = "train_dataloader"

dl_cfg = cfg.data[DATALOADER_CONFIG_NAME]
dataset_config = instantiate(dl_cfg.dataset_config)
dataset_class = instantiate(dl_cfg.dataset_class)

files = find_tfrecord_files(Path(dl_cfg.data_folder), ".", limit=3) if Path(dl_cfg.data_folder).name == "." else sorted(Path(dl_cfg.data_folder).glob("*.tfrecord.gz"))[:3]
files = [str(path) for path in files]
print(f"Using data folder: {dl_cfg.data_folder}")
print(f"Using {len(files)} file(s):")
for file in files:
    print(" -", file)

dataset = dataset_class(
    dataset_config=dataset_config,
    data_folder=dl_cfg.data_folder,
    list_of_file_paths=files,
    global_rank=0,
    is_for_training=False,
)
preprocessed_samples = list(itertools.islice(iter(dataset), NUM_PREPROCESSED_SAMPLES))
print(f"Loaded {len(preprocessed_samples)} preprocessed sample(s).")

In [ ]:
print("Preprocessed sample structure:")
pprint(describe_structure(preprocessed_samples[0]))

print("\nPreprocessed sample preview:")
pprint(preview_value(preprocessed_samples[0]))

## 3. 可选：查看 collate 后 batch 结构

如果 experiment 的 dataloader 配置有 `collate_fn`，可以抽样若干条预处理后的 row 并查看 collate 输出结构。

In [ ]:
collate_fn = instantiate(dl_cfg.collate_fn)
batch = collate_fn(
    preprocessed_samples,
    labels=instantiate(dl_cfg.labels),
    sequence_length=dl_cfg.sequence_length,
    masking_token=dl_cfg.masking_token,
    padding_token=dl_cfg.padding_token,
)

print("Collated batch structure:")
pprint(describe_structure(batch))

print("\nCollated batch preview:")
pprint(preview_value(batch))